## 9. KlG-Akzeptanz → Gemeinde-Raster

In [ ]:
print("=== Akzeptanz-Layer ===")
muni = gpd.read_file(RAW / "swissboundaries/swissboundaries_gemeinden.shp")
canton_col = next(c for c in muni.columns if "KANT" in c.upper() or c.upper().startswith("KT"))
name_col = next(c for c in muni.columns if "NAME" in c.upper())
gr_muni = muni[muni[canton_col].astype(str).str.contains("GR", case=False, na=False)].copy()

df_klg = pd.read_json(RAW / "acceptance/klg_2023.json")
df_klg["yes_share"] = df_klg["yeas"] / (df_klg["yeas"] + df_klg["nays"])
print(f"  KlG: {len(df_klg)} Gemeinden, Ja-Anteil {df_klg['yes_share'].min():.2f}–{df_klg['yes_share'].max():.2f}")

gr_muni["_k"] = gr_muni[name_col].str.strip().str.upper()
votes = df_klg.assign(_k=df_klg["name"].str.strip().str.upper())
gdf_acc = gr_muni.merge(votes[["_k", "yes_share"]], on="_k", how="left")
gdf_acc["yes_share"] = gdf_acc["yes_share"].fillna(gdf_acc["yes_share"].median())
print(f"  Join: {gdf_acc['yes_share'].notna().sum()}/{len(gdf_acc)} Gemeinden")

shapes = [(g, v) for g, v in zip(gdf_acc.geometry, gdf_acc["yes_share"]) if g is not None]
acc = rasterize(shapes, out_shape=(height, width), transform=transform, fill=NODATA, dtype=np.float32)
with rasterio.open(PROC / "criteria/acceptance_klg_gr_25m.tif", "w", **ref_profile) as dst:
    dst.write(acc, 1)
v_a = acc[valid & (acc != NODATA)]
print(f"  Akzeptanz: {v_a.min():.2f}–{v_a.max():.2f}")

## 10. Übersicht & Vorschau

In [ ]:
print("=== Erzeugte Dateien ===\n")
for folder in ["criteria", "distances"]:
    files = sorted((PROC / folder).glob("*.tif"))
    if files:
        print(f"{folder}/")
        for f in files:
            print(f"  {f.name:40s} {f.stat().st_size/1e6:>6.1f} MB")
        print()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.suptitle("Normalisierte Eignungsfaktoren [0, 1]", fontsize=13, fontweight="bold")
criteria = sorted((PROC / "criteria").glob("f*.tif"))
for ax, fp in zip(axes.flat, criteria):
    with rasterio.open(fp) as src:
        data = np.ma.masked_equal(src.read(1), NODATA)
    ax.imshow(data, cmap="YlOrRd", vmin=0, vmax=1, aspect="equal")
    ax.set_title(fp.stem, fontsize=8); ax.set_axis_off()
for ax in axes.flat[len(criteria):]:
    ax.set_visible(False)
plt.tight_layout()
fig.savefig(OUT / "wlc_criteria_overview.png", dpi=150, bbox_inches="tight")